# 02 — Población, ventana y target (definición reproducible)

**Unidad:** usuario–vehículo–ventana. **Ancla:** cada mantenimiento programado completado en la red (o el inicio de garantía
para el primer service). **Vencimiento:** `due = ancla + min(365 días, K_gen / tasa_uso)` con K = 15.000 km (P703) /
10.000 km (P375) y tasa de uso = km al ancla / edad del vehículo. **Apertura de ventana y scoring:** `due − 30 días`.
**Horizonte:** `due + 90 días`. **Target:** `churn = 1` si no hay mantenimiento programado completado entre la apertura y el
cierre del horizonte. Ventanas cuyo horizonte no cerró antes del cutoff quedan **censuradas** (no se usan para entrenar;
son la población a scorear hoy). Implementación: `src/repurchase/ventanas.py`; parámetros: `config/params.json`.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
from IPython.display import Image, Markdown, display
from repurchase import config
from repurchase.eventos import appointments, CUTOFF
from repurchase.io import load_sales, load_agenda
print("cutoff de datos:", CUTOFF.date())

cutoff de datos: 2026-08-25


In [2]:
from repurchase.ventanas import WindowParams, build_windows
cfg = json.loads((ROOT / "config/params.json").read_text(encoding="utf8"))
params = WindowParams.from_dict(cfg["ventana"]); print(params.to_dict())
win = build_windows(params=params)
display(win.status.value_counts().to_frame("ventanas"))

{'cycle_days': 365, 'km_interval': 16000, 'km_by_generation': {'RANGER (P703)': 16000, 'RANGER RAPTOR (P703)': 16000, 'RANGER (P375)': 10000, 'RANGER RAPTOR': 10000, 'RANGER': 10000}, 'lead_days': 30, 'horizon_days': 90, 'min_gap_days': 90, 'first_due_days': 300, 'anchor': 'last_maintenance', 'km_source': 'current', 'min_history_days': 0, 'split_visit_days': 30, 'label_margin_days': 30, 'label_mode': 'ventana'}


,ventanas
status,
evaluable,142637
preempted,59697
futura,59368
fuera_de_rango,50998
censurada,35190


Estados: `evaluable` (label conocido), `censurada` (horizonte abierto al cutoff), `preempted` (el vehículo volvió antes
de que abriera la ventana: nunca estuvo en ventana, no se scorea), `fuera_de_rango` (scoring antes de 2024), `futura`
(scoring después del cutoff).

In [3]:
ev = win[win.status == "evaluable"]
print(f"ventanas evaluables: {len(ev):,}   vehículos: {ev.vehicle_id.nunique():,}   churn: {ev.label_churn.mean():.1%}")
display(ev.groupby(ev.scoring_date.dt.to_period("Q")).label_churn.agg(["size", "mean"]).rename(columns={"size": "ventanas", "mean": "churn"}))
display(ev.groupby("binding_rule").label_churn.agg(["size", "mean"]).rename(columns={"size": "ventanas", "mean": "churn"}))
display(ev.groupby("anchor_type").label_churn.agg(["size", "mean"]).rename(columns={"size": "ventanas", "mean": "churn"}))

ventanas evaluables: 142,637   vehículos: 80,276   churn: 41.6%


,ventanas,churn
scoring_date,,
2024Q1,4463,0.440511
2024Q2,12295,0.392924
2024Q3,14553,0.399024
2024Q4,15318,0.402533
2025Q1,17221,0.419720
2025Q2,18776,0.423945
2025Q3,19414,0.426702
2025Q4,20888,0.416028
2026Q1,19709,0.426201


,ventanas,churn
binding_rule,,
km,87690,0.411712
piso,16392,0.274768
primer_service_tiempo,31760,0.481014
tiempo,6795,0.507432


,ventanas,churn
anchor_type,,
maintenance,110877,0.397332
warranty,31760,0.481014


## Chequeo anti-leakage: nada posterior al scoring entra como feature

In [4]:
from repurchase.features import build_features
f = build_features(ev.sample(20000, random_state=0))
# Todas las recencias son positivas: el último evento usado es estrictamente anterior al scoring.
chk = {c: int((f[c] <= 0).sum()) for c in ["days_since_last_appt", "days_since_last_maint", "days_since_last_visit", "days_since_last_survey"]}
print("features de recencia con valor <= 0 (debe ser 0 en todas):", chk)
print("ventanas donde la primera lectura de km posterior al scoring se filtró: 0 por construcción (merge_asof backward estricto)")

features de recencia con valor <= 0 (debe ser 0 en todas): {'days_since_last_appt': 0, 'days_since_last_maint': 0, 'days_since_last_visit': 0, 'days_since_last_survey': 0}
ventanas donde la primera lectura de km posterior al scoring se filtró: 0 por construcción (merge_asof backward estricto)


## Sensibilidad de la definición a sus parámetros

In [5]:
p = ROOT / "reports/modelo/sensibilidad_ventana.csv"
if p.exists():
    s = pd.read_csv(p)
    display(s[["km_by_generation", "lead_days", "horizon_days", "first_due_days", "anchor", "km_source", "n_evaluables", "churn_rate", "pct_preempted", "pct_piso", "captura_retornos_18m", "mediana_dias_retorno_vs_due"]].round(3))
else:
    print("correr scripts/sensibilidad_ventana.py")

,km_by_generation,lead_days,horizon_days,first_due_days,anchor,km_source,n_evaluables,churn_rate,pct_preempted,pct_piso,captura_retornos_18m,mediana_dias_retorno_vs_due
0,(P703)=16000|(P703)=16000|(P375)=10000|RAPTOR=...,30,60,300,last_maintenance,current,149423,0.507,0.201,0.112,0.437,8.0
1,(P703)=16000|(P703)=16000|(P375)=10000|RAPTOR=...,30,90,300,last_maintenance,current,142558,0.416,0.201,0.115,0.506,16.0
2,(P703)=16000|(P703)=16000|(P375)=10000|RAPTOR=...,30,120,300,last_maintenance,current,135824,0.365,0.201,0.116,0.538,20.0
3,(P703)=16000|(P703)=16000|(P375)=10000|RAPTOR=...,30,180,300,last_maintenance,current,122114,0.306,0.201,0.121,0.562,25.0
4,(P703)=16000|(P703)=16000|(P375)=10000|RAPTOR=...,60,60,300,last_maintenance,current,166877,0.449,0.132,0.121,0.555,-1.0
5,(P703)=16000|(P703)=16000|(P375)=10000|RAPTOR=...,60,90,300,last_maintenance,current,159220,0.369,0.132,0.124,0.625,5.0
6,(P703)=16000|(P703)=16000|(P375)=10000|RAPTOR=...,60,120,300,last_maintenance,current,151678,0.324,0.132,0.126,0.659,9.0
7,(P703)=16000|(P703)=16000|(P375)=10000|RAPTOR=...,60,180,300,last_maintenance,current,136393,0.271,0.132,0.131,0.687,14.0
8,(P703)=15000|(P703)=15000|(P375)=10000,30,60,300,last_maintenance,current,152646,0.509,0.193,0.119,0.443,9.0
9,(P703)=15000|(P703)=15000|(P375)=10000,30,90,300,last_maintenance,current,145529,0.416,0.193,0.121,0.516,17.0
